# Etapa 2
## Criação de modelos alunos e destilação de treinamento

In [2]:
import torch
from torch import nn
import torchvision
import numpy as np
import torchvision
import matplotlib.pyplot as plt
import pandas as pd
import torch.nn.functional as F


torch.backends.cudnn.benchmark = True            # autotuner p/ input fixo
torch.backends.cuda.matmul.allow_tf32 = True     # TF32 no Ampere
torch.backends.cudnn.allow_tf32 = True           # TF32 no Ampere
device = torch.device("cuda" if torch.cuda.is_available() else Exception("No GPU available"))


In [ ]:
class Student(nn.Module):
    def __init__(self, num_classes: int, teacher_dim: int):
        super(Student, self).__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False), # 3x224x224 -> 64x112x112
            nn.BatchNorm2d(64),
            nn.GELU(), 
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)                  # 64x112x112 -> 64x56x56
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False), # 64x56x56 -> 128x28x28
            nn.BatchNorm2d(128),
            nn.GELU(),
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=False), # 128x28x28 -> 128x28x28
            nn.BatchNorm2d(128),
            nn.GELU(),
        )

        self.layer4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1, bias=False), # 128x28x28 -> 256x14x14
            nn.BatchNorm2d(256),
            nn.GELU(),
        )

        self.layer5 = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=1, bias=False), # 256x14x14 -> 256x7x7
            nn.BatchNorm2d(256),
            nn.GELU(),
        )

        self.layer6 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1, bias=False), # 256x7x7 -> 512x4x4
            nn.BatchNorm2d(512),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))        # 512x4x4 -> 512x1x1
        self.flatten = nn.Flatten(1, -1)                   # 512x1x1 -> 512

        self.proj = nn.Linear(512, teacher_dim)           # 512 -> teacher_dim
        self.classifier = nn.Linear(teacher_dim, num_classes) # teacher_dim -> num

    def encode(self, x):
        x = self.layer6(self.layer5(self.layer4(self.layer3(self.layer2(self.layer1(x))))))
        x = self.avgpool(x)
        return self.flatten(x)
    
    def project(self, x):
        return self.proj(self.encode(x))
    
    def forward(self, x):
        return self.classifier(self.proj(self.encode(x)))

In [ ]:
cifar_100_students = {
    "resnet50" : Student(num_classes=100, teacher_dim=2048),
    "convnext_base" : Student(num_classes=100, teacher_dim=1024),
    "vgg16" : Student(num_classes=100, teacher_dim=512)
} 


oxford_pets_students = {
    "resnet50" : Student(num_classes=37, teacher_dim=2048),
    "convnext_base" : Student(num_classes=37, teacher_dim=1024),
    "vgg16" : Student(num_classes=37, teacher_dim=512)
}

# Baixando os Datasets e carregando em DataLoader

### Baixando o dataset cifar100

In [ ]:
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# duas visoes do MESMO conjunto de treino: aug (treino) e limpa (validacao)
cifar_train_full = torchvision.datasets.CIFAR100(root="./data", train=True, download=False, transform=train_transform)
cifar_eval_full  = torchvision.datasets.CIFAR100(root="./data", train=True, download=False, transform=test_transform)

g = torch.Generator()
perm = torch.randperm(len(cifar_train_full), generator=g).tolist()
cut = int(0.8 * len(perm))
train_idx, val_idx = perm[:cut], perm[cut:]

train_set_CIFAR100 = Subset(cifar_train_full, train_idx)   # com augmentation
val_set_CIFAR100   = Subset(cifar_eval_full,  val_idx)     # sem augmentation
test_set_CIFAR100  = torchvision.datasets.CIFAR100(root="./data", train=False, download=False, transform=test_transform)

train_loader_CIFAR100 = DataLoader(train_set_CIFAR100, batch_size=256, shuffle=True,  num_workers=4, pin_memory=True)
val_loader_CIFAR100   = DataLoader(val_set_CIFAR100,   batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
test_loader_CIFAR100  = DataLoader(test_set_CIFAR100,  batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

print(f"CIFAR-100 -> treino {len(train_set_CIFAR100)} | val {len(val_set_CIFAR100)} | teste {len(test_set_CIFAR100)}")

### Baixando o dataset oxfordpet

In [ ]:
oxford_train_full = torchvision.datasets.OxfordIIITPet(root="./data", download=False, transform=train_transform)
oxford_eval_full  = torchvision.datasets.OxfordIIITPet(root="./data", download=False, transform=test_transform)

g = torch.Generator()
perm = torch.randperm(len(oxford_train_full), generator=g).tolist()
cut = int(0.8 * len(perm))
train_idx, val_idx = perm[:cut], perm[cut:]

train_set_oxford102 = Subset(oxford_train_full, train_idx)   # com augmentation
val_set_oxford102   = Subset(oxford_eval_full,  val_idx)     # sem augmentation
test_set_oxford102  = torchvision.datasets.OxfordIIITPet(root="./data", download=False, split="test", transform=test_transform)

train_loader_oxford102 = DataLoader(train_set_oxford102, batch_size=64, shuffle=True,  num_workers=4, pin_memory=True)
val_loader_oxford102   = DataLoader(val_set_oxford102,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader_oxford102  = DataLoader(test_set_oxford102,  batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"Oxford-IIIT Pet -> treino {len(train_set_oxford102)} | val {len(val_set_oxford102)} | teste {len(test_set_oxford102)}")

# Carregando os modelos professores

In [35]:
vgg_weights = torchvision.models.VGG16_Weights.DEFAULT
vgg_16 = torchvision.models.vgg16(weights=vgg_weights)

resnet_weights = torchvision.models.ResNet50_Weights.DEFAULT
resnet_50 = torchvision.models.resnet50(weights=resnet_weights)

convnext_weights = torchvision.models.ConvNeXt_Base_Weights.DEFAULT
convnext_base = torchvision.models.convnext_base(weights=convnext_weights)

models = {
    "vgg16": vgg_16,
    "resnet50": resnet_50,
    "convnext_base": convnext_base
}

In [ ]:
import copy

vgg_16_cifar100 = copy.deepcopy(models["vgg16"])
resnet_50_cifar100 = copy.deepcopy(models["resnet50"])
convnext_base_cifar100 = copy.deepcopy(models["convnext_base"])

vgg_16_cifar100.classifier[6] = torch.nn.Linear(in_features=4096, out_features=100, bias=True)
vgg_16_cifar100.load_state_dict(torch.load("./models/vgg16_cifar100.pth"))

resnet_50_cifar100.fc = torch.nn.Linear(in_features=2048, out_features=100, bias=True)
resnet_50_cifar100.load_state_dict(torch.load("./models/resnet50_cifar100.pth"))

convnext_base_cifar100.classifier[2] = torch.nn.Linear(in_features=1024, out_features=100, bias=True)
convnext_base_cifar100.load_state_dict(torch.load("./models/convnext_base_cifar100.pth"))



vgg_16_oxford102 = copy.deepcopy(models["vgg16"])
resnet_50_oxford102 = copy.deepcopy(models["resnet50"])
convnext_base_oxford102 = copy.deepcopy(models["convnext_base"])

vgg_16_oxford102.classifier[6] = torch.nn.Linear(in_features=4096, out_features=102, bias=True)
vgg_16_oxford102.load_state_dict(torch.load("./models/vgg16_oxford102.pth"))

resnet_50_oxford102.fc = torch.nn.Linear(in_features=2048, out_features=102, bias=True)
resnet_50_oxford102.load_state_dict(torch.load("./models/resnet50_oxford102.pth"))

convnext_base_oxford102.classifier[2] = torch.nn.Linear(in_features=1024, out_features=102, bias=True)
convnext_base_oxford102.load_state_dict(torch.load("./models/convnext_base_oxford102.pth"))

<All keys matched successfully>

In [38]:
cifar_100_teachers = {
    "resnet50" : resnet_50_cifar100,
    "convnext_base" : convnext_base_cifar100,
    "vgg16" : vgg_16_cifar100
}

oxford_pets_teachers = {
    "resnet50" : resnet_50_oxford102,
    "convnext_base" : convnext_base_oxford102,
    "vgg16" : vgg_16_oxford102
}

# Treinamento - Fase 1

In [ ]:
import os
os.makedirs("./models", exist_ok=True)


@torch.no_grad()
def teacher_encode(model: nn.Module, x, kind: str):
    if kind == "resnet50":
        x = model.maxpool(model.relu(model.bn1(model.conv1(x))))
        x = model.layer4(model.layer3(model.layer2(model.layer1(x))))
        x = model.avgpool(x)
    elif kind == "convnext_base":
        x = model.avgpool(model.features(x))
    elif kind == "vgg16":
        x = F.adaptive_avg_pool2d(model.features(x), 1)
    else:
        raise ValueError(f"Unknown model kind: {kind}")
    return torch.flatten(x, 1)


@torch.no_grad()
def eval_distill_mse(student, teacher, loader, kind, device):
    """MSE (normalizada) media no conjunto de validacao. Nao altera pesos."""
    student.eval()
    total = 0.0
    n = 0
    for x, _ in loader:
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            ft = teacher_encode(teacher, x, kind)
            fs = student.project(x)
        loss = F.mse_loss(F.normalize(fs.float(), dim=1), F.normalize(ft.float(), dim=1))
        total += loss.item() * x.size(0)
        n += x.size(0)
    return total / n


def destill_features(student: Student, teacher: nn.Module, train_loader, val_loader,
                     kind: str, epochs: int, save_path: str, lr: float = 1e-3):
    teacher.to(device).eval()
    for param in teacher.parameters():
        param.requires_grad = False

    student.to(device)
    encoder_parameters = [param for name, param in student.named_parameters() if "classifier" not in name]
    optimizer = torch.optim.AdamW(encoder_parameters, lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda")
    best_val_mse = float("inf")

    for epoch in range(epochs):
        student.train()
        run = 0.0
        n = 0
        for x, _ in train_loader:
            x = x.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                ft = teacher_encode(teacher, x, kind)
                fs = student.project(x)
            loss = F.mse_loss(F.normalize(fs.float(), dim=1), F.normalize(ft.float(), dim=1))
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run += loss.item() * x.size(0)
            n += x.size(0)
        scheduler.step()

        train_mse = run / n
        val_mse = eval_distill_mse(student, teacher, val_loader, kind, device)
        if val_mse < best_val_mse:                       # menor val_MSE -> salva
            best_val_mse = val_mse
            torch.save(student.state_dict(), save_path)
        print(f"{kind}] epoch {epoch:02d} | train_mse {train_mse:.5f} | val_mse {val_mse:.5f} | "
              f"lr {scheduler.get_last_lr()[0]:.2e}")

    # recarrega o MELHOR encoder p/ a memoria (a Fase 2 deve partir do melhor, nao do ultimo)
    student.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    print(f"melhor val_MSE = {best_val_mse:.5f} (encoder recarregado de {save_path})\n")
    return best_val_mse

#### Treinando encoder de alunos do CIFAR100

In [ ]:
for name, teacher in cifar_100_teachers.items():
    print(f"=== Fase 1 | CIFAR-100 | aluno de {name} ===")
    destill_features(cifar_100_students[name], teacher, train_loader_CIFAR100, val_loader_CIFAR100,
                     name, epochs=15, save_path=f"./models/{name}_student_cifar100_enc.pth")

#### Treinando encoder de alunos do OxfordIIIT pet

In [ ]:
for name, teacher in oxford_pets_teachers.items():
    print(f"=== Fase 1 | Oxford | aluno de {name} ===")
    destill_features(oxford_pets_students[name], teacher, train_loader_oxford102, val_loader_oxford102,
                     name, epochs=15, save_path=f"./models/{name}_student_oxford102_enc.pth")

# Treinamento - Fase 2

In [ ]:
@torch.no_grad()
def evaluate_cls(model, loader, device):
    """Loss (CE) e acuracia na validacao."""
    model.eval()
    loss_sum = 0.0
    correct = 0
    n = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = model(x)
            loss = F.cross_entropy(logits, y)
        loss_sum += loss.item() * x.size(0)
        correct  += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    return loss_sum / n, correct / n


def train_classifier(student: Student, train_loader, val_loader, epochs: int,
                     dataset_name: str, model_name: str, lr: float = 1e-3):
    student.to(device)

    # parte do MELHOR encoder da Fase 1 (robusto a restart do kernel)
    enc_path = f"./models/{model_name}_student_{dataset_name}_enc.pth"
    student.load_state_dict(torch.load(enc_path, map_location=device, weights_only=True))

    # congela encoder + proj; so o classifier treina
    for pname, param in student.named_parameters():
        param.requires_grad = ("classifier" in pname)
    student.eval()   # encoder congelado: BN fixo, dropout off (classifier e Linear, indiferente)

    optimizer = torch.optim.AdamW(student.classifier.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", init_scale=1024)
    best_val_acc = -1.0
    save_path = f"./models/{model_name}_student_{dataset_name}.pth"

    for epoch in range(epochs):
        run = 0.0
        n = 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                logits = student(x)
                loss = F.cross_entropy(logits, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run += loss.item() * x.size(0)
            n += x.size(0)
        scheduler.step()

        train_loss = run / n
        val_loss, val_acc = evaluate_cls(student, val_loader, device)
        if val_acc > best_val_acc:                       # maior val_acc -> salva
            best_val_acc = val_acc
            torch.save(student.state_dict(), save_path)
        print(f"epoch {epoch:02d} | train_loss {train_loss:.4f} | "
              f"val_loss {val_loss:.4f} acc {val_acc:.4f} | lr {scheduler.get_last_lr()[0]:.2e}")

    print(f" melhor val_acc = {best_val_acc:.4f} (salvo em {save_path})\n")
    return best_val_acc


#### Treinando classificador de alunos do OxfordIIIT pet

In [4]:
for name, student in oxford_pets_students.items():
    print(f"=== Fase 2 | Oxford | aluno de {name} ===")
    train_classifier(student, train_loader_oxford102, val_loader_oxford102,
                     epochs=15, dataset_name="oxford102", model_name=name)

NameError: name 'oxford_pets_students' is not defined

#### Treinando classificador de alunos do CIFAR100

In [ ]:
for name, student in cifar_100_students.items():
    print(f"=== Fase 2 | CIFAR-100 | aluno de {name} ===")
    train_classifier(student, train_loader_CIFAR100, val_loader_CIFAR100,
                     epochs=15, dataset_name="cifar100", model_name=name)

# Avaliação dos modelos alunos

In [ ]:
@torch.no_grad()
def evaluate(name, model, loader, n_samples, criterion, device):
    model.eval()
    running_loss = torch.zeros((), device=device)
    correct = torch.zeros((), device=device)

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.detach() * images.size(0)   # média ponderada
        correct += (outputs.argmax(1) == labels).sum()

    avg_loss = (running_loss / n_samples).item()
    accuracy = (correct / n_samples).item()

    print(f"Model: {name} | Test Loss: {avg_loss:.4f} | Test Acc: {accuracy:.4f}")
    return avg_loss, accuracy


def load_checkpoint(model, path, device):
    """Recarrega os pesos salvos em disco (robusto a crash/restart do kernel)."""
    state_dict = torch.load(path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    return model.to(device)


criterion = nn.CrossEntropyLoss()
accuracy = {}

print("=== CIFAR-100 ===")
for name, model in cifar_100_students.items():
    model = load_checkpoint(model, f"./models/{name}_student_cifar100.pth", device)
    _, test_acc = evaluate(name, model, test_loader_CIFAR100,
                           len(test_set_CIFAR100), criterion, device)
    accuracy[f"{name}_cifar100"] = test_acc

print("=== Oxford-IIIT Pet ===")
for name, model in oxford_pets_students.items():
    model = load_checkpoint(model, f"./models/{name}_student_oxford102.pth", device)
    _, test_acc = evaluate(name, model, test_loader_oxford102,
                           len(test_set_oxford102), criterion, device)
    accuracy[f"{name}_oxford102"] = test_acc


In [ ]:
accuracy_csv = pd.DataFrame.from_dict(accuracy, orient="index", columns=["accuracy"])
accuracy_csv.to_csv("./results/student.csv")